# 03 · Data preparation

**Input:** `reviews_clean.pkl`. **Output:** `train.pkl`, `val.pkl`, `test.pkl`, `class_weights.json`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

In [ ]:
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.config import CFG, LABELS, SEED, setup_notebook
from src.data import CLASS_WEIGHTS_PATH, SPLIT_PATHS, load_clean
from src.text import build_input_text

setup_notebook()
df = load_clean()
if CFG['sample_size']:
    df = df.sample(n=CFG['sample_size'], random_state=SEED).reset_index(drop=True)
    print(f'QUICK_RUN: sampled {len(df):,} reviews')
print(f'{len(df):,} review')

## 1. Build the model input text

In [ ]:
t0 = time.time()
df['text'] = build_input_text(df['Summary'], df['Text'])
print(f'Cleaned {len(df):,} reviews in {time.time() - t0:.1f}s')
df[['Summary', 'Text', 'text', 'sentiment']].head(3)

## 2. Train / validation / test split (80/10/10, stratified)

In [ ]:
cols = ['text', 'label', 'Score']
train_df, temp_df = train_test_split(df[cols], test_size=0.2, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
test_df['n_words'] = test_df['text'].str.split().str.len()

split_summary = pd.DataFrame({name: d['label'].value_counts(normalize=True).reindex([0, 1, 2]).values
                              for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]},
                             index=LABELS).T
split_summary.insert(0, 'Reviews', [len(train_df), len(val_df), len(test_df)])
display(split_summary.style.format({'Reviews': '{:,}', **{l: '{:.2%}' for l in LABELS}}))

## 3. Class weights

In [ ]:
y_train = train_df['label'].values
class_counts = np.bincount(y_train, minlength=3)
class_weights = len(y_train) / (3 * class_counts)
pd.DataFrame({'Train samples': class_counts, 'Weight': class_weights.round(3)}, index=LABELS)

## 4. Save

In [ ]:
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_pickle(SPLIT_PATHS[name])
CLASS_WEIGHTS_PATH.write_text(json.dumps({'labels': LABELS, 'counts': class_counts.tolist(),
                                          'weights': class_weights.tolist()}, indent=2), encoding='utf-8')
for p in [*SPLIT_PATHS.values(), CLASS_WEIGHTS_PATH]:
    print('Saved', p)